In [1]:
import pandas as pd
import yaml
import importlib
import etl.normalization as normalization
importlib.reload(normalization)
from etl.typecasting import apply_schema_types, verify_schema_types
from etl.validator import add_range_and_allowed_flags, add_mandatory_flags

In [2]:
with open(r"hf_schema.yaml", "r", encoding="utf-8") as f:
    schema = yaml.safe_load(f)

schema

{'version': '1.0',
 'schema_id': 'heatflow:v1',
 'name': 'heat_flow_db_complete',
 'normalization': {'string': {'trim': True,
   'collapse_space': True,
   'case_insensitive': True,
   'enforce_brackets': True,
   'normalize_separator': True,
   'missing_tokens': ['', '-', 'NA', '<NA>', 'none', 'null']},
  'numeric': {'decimal_comma_to_dot': True,
   'strip_thousands_separators': [' '],
   'missing_tokens': ['', 'NA', '<NA>', '-']}},
 'core': {'ID': {'dtype': 'int64', 'unique': True, 'min': 1},
  'Obligation': {'dtype': 'string', 'allowed': ['M', 'R', 'O', '-']},
  'Level': {'dtype': 'string', 'allowed': ['Parent', 'Child', 'Admin']}},
 'columns': {'P1': {'dtype': 'float64',
   'range': [-999999.9, 999999.9],
   'obligation': 'M',
   'comment': 'HF Value'},
  'P2': {'dtype': 'float64',
   'range': [0.0, 999999.9],
   'obligation': 'M',
   'comment': 'HF Uncertainty'},
  'P3': {'dtype': 'string', 'obligation': 'M', 'comment': 'Name'},
  'P4': {'dtype': 'float64',
   'range': [-90.0, 90.

In [3]:
df_raw = pd.read_excel(
    r"./testing/Abc_xyz_2008.xlsx",
    sheet_name=1,
    header=0,
    dtype=str 
)
print(df_raw.shape)
df_raw.head(10)

#check if everything is string 
df_raw.dtypes

(28, 71)


/opt/homebrew/Caskroom/miniconda/base/envs/thermo-env/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


ID    object
P1    object
P2    object
P3    object
P4    object
       ...  
A4    object
A5    object
A6    object
A7    object
A8    object
Length: 71, dtype: object

In [4]:
# Tranfrom Excel to Parquet (all to string)
#divide into meta and data
df_raw["row_type"] = ["meta"] * 7 + ["data"] * (len(df_raw) - 7)
df_raw_str = df_raw.astype("string[pyarrow]")
#convert to parquet
df_raw_str.to_parquet(
    r"./testing/Abc_xyz_2008_raw.parquet",
    index=False)



In [5]:
# Load Parquet in as String 
df_raw_parquet_string = pd.read_parquet(
    r"./testing/Abc_xyz_2008_raw.parquet",
    dtype_backend="pyarrow"
).astype("string[pyarrow]")  
df_raw_parquet_string 

df_raw_parquet_string.dtypes

ID          string[pyarrow]
P1          string[pyarrow]
P2          string[pyarrow]
P3          string[pyarrow]
P4          string[pyarrow]
                 ...       
A5          string[pyarrow]
A6          string[pyarrow]
A7          string[pyarrow]
A8          string[pyarrow]
row_type    string[pyarrow]
Length: 72, dtype: object

In [6]:
df_meta, df_data_norm = normalization.normalize_only_data_rows(
    df_raw_parquet_string,
    schema,
)

df_data_norm.dtypes

df_meta

,ID,P1,P2,P3,P4,P5,P6,P7,P8,P9,...,C49,A1,A2,A3,A4,A5,A6,A7,A8,row_type
0,Obligation,M,M,M,M,M,M,M,R,R,...,O,-,-,-,-,-,-,-,-,meta
1,Domain,"B,S","B,S","B,S","B,S","B,S","B,S","B,S","B,S","B,S",...,"B,S",-,-,-,-,-,-,-,-,meta
2,Quality Relevance,U score,U score,-,-,-,M score,-,-,-,...,-,-,-,-,-,-,-,-,-,meta
3,Name,Heat-flow value,Heat-flow uncertainty,Site name,Geographical latitude,Geographical longitude,Elevation (Geographical),Basic geographical environment,General comments parent level,Flag heat production of the overburden (heat-f...,...,IGSN,Reviewer_name,Reviewer_comment,Review Date,Country,Region,Continent,Domain,Unique entry ID,meta
4,Short Name,q,q_uncertainty,name,lat_NS,long_EW,elevation,environment,p_comment,corr_HP_flag,...,Ref_ISGN,Reviewer_name,Reviewer_comment,Review_date,Country,Region,Continent,Domain,ID,meta
5,Unit,mW/m²,mW/m²,-,degrees,degrees,m,-,-,-,...,-,-,-,-,-,-,-,-,-,meta
6,Allowed range of values,"-999,999.9 – 999,999.9","0 – 999,999.9",-,-90.00000 – +90.00000,-180.00000 – +180.00000,-12000 – +9000,[Onshore (continental)],-,Yes,...,-,-,-,-,-,-,-,-,-,meta


In [7]:
# 2) type-cast only data rows
df_data_typed = apply_schema_types(df_data_norm, schema)



# 4) now dtypes here are your ground truth
print(df_data_typed.dtypes)
warns = verify_schema_types(df_data_typed, schema).query("status == 'OK'")
print(warns)

ID                    Int64
P1                  float64
P2                  float64
P3          string[pyarrow]
P4                  float64
                 ...       
A5          string[pyarrow]
A6          string[pyarrow]
A7          string[pyarrow]
A8          string[pyarrow]
row_type    string[pyarrow]
Length: 72, dtype: object
   column expected   actual status
0      ID    int64    int64     OK
3      P1  float64  float64     OK
4      P2  float64  float64     OK
5      P3   string   string     OK
6      P4  float64  float64     OK
..    ...      ...      ...    ...
63     A4   string   string     OK
64     A5   string   string     OK
65     A6   string   string     OK
66     A7  string,   string     OK
67     A8   string   string     OK

[66 rows x 4 columns]


In [8]:
print(df_data_typed.dtypes.tail(10))   
#print(df_typed.head(5))

C49         string[pyarrow]
A1          string[pyarrow]
A2          string[pyarrow]
A3           datetime64[ns]
A4          string[pyarrow]
A5          string[pyarrow]
A6          string[pyarrow]
A7          string[pyarrow]
A8          string[pyarrow]
row_type    string[pyarrow]
dtype: object


In [9]:
df_data_typed["P7"]


7      [onshore-(continental)]
8      [onshore-(continental)]
9     [offshore-(continental)]
10         [offshore-(marine)]
11               [unspecified]
12     [onshore-(continental)]
13         [offshore-(marine)]
14     [onshore-(continental)]
15     [onshore-(continental)]
16     [onshore-(continental)]
17         [offshore-(marine)]
18         [offshore-(marine)]
19         [offshore-(marine)]
20         [offshore-(marine)]
21     [onshore-(continental)]
22     [onshore-(continental)]
23     [onshore-(continental)]
24     [onshore-(continental)]
25     [onshore-(continental)]
26     [onshore-(continental)]
27     [onshore-(continental)]
Name: P7, dtype: string

In [10]:
df_checked_mandatory = add_mandatory_flags(df_data_typed, schema)
df_checked_mandatory

,ID,P1,P2,P3,P4,P5,P6,P7,P8,P9,...,C32__missing,C38__missing,C39__missing,C41__missing,C42__missing,C43__missing,C44__missing,C45__missing,C46__missing,C47__missing
7,1,39.35592,3.34944,chicheng 13,40.683333,115.500000,1300.0,[onshore-(continental)],<NA>,[no],...,False,False,False,False,False,False,False,False,True,False
8,2,33.49440,NaN,fanshan 103,40.200000,115.433333,750.0,[onshore-(continental)],<NA>,[unspecified],...,False,False,False,False,False,False,False,False,True,False
9,3,26.40000,0.65000,fangshan 46,40.166667,115.433333,875.0,[offshore-(continental)],mean hf,[unspecified],...,False,False,False,False,False,False,False,False,True,False
10,4,26.40000,0.65000,fangshan 46,40.166667,115.433333,875.0,[offshore-(marine)],mean hf,[unspecified],...,True,False,False,False,False,False,False,False,True,False
11,5,30.14496,NaN,chengde 10,40.583333,117.850000,NaN,[unspecified],<NA>,[unspecified],...,False,False,False,False,False,False,False,False,True,False
12,6,50.66028,3.34944,yanging 72-7,40.400000,116.266667,NaN,[onshore-(continental)],<NA>,[unspecified],...,False,False,False,False,False,False,False,False,True,False
13,7,56.52180,4.18680,yanging 72-5,40.416667,116.250000,528.0,[offshore-(marine)],<NA>,[unspecified],...,True,False,False,False,False,False,False,False,True,False
14,8,77.03712,1.67472,yanging 2,40.450000,115.933333,528.0,[onshore-(continental)],<NA>,[no],...,False,False,False,False,False,False,False,False,False,False
15,9,38.50000,NaN,caraiba,-9.470000,-39.835800,400.0,[onshore-(continental)],mean hf,[unspecified],...,False,False,False,False,False,False,False,False,False,True
16,10,51.00000,12.00000,jacobina,-11.250000,-40.500000,850.0,[onshore-(continental)],<NA>,[unspecifiedd],...,False,False,False,False,False,False,False,False,False,True


In [13]:
df_checked = add_range_and_allowed_flags(df_data_typed, schema)


In [14]:
col = "P9"

# what was in your schema
allowed_raw = pd.Series(schema["columns"][col]["allowed"], dtype="string")

# normalized the same way as validator does
from etl.normalization import normalize_vocabulary_series
allowed_norm = normalize_vocabulary_series(allowed_raw)
allowed_set = {str(a).strip().lower() for a in allowed_norm.dropna()}

print("ALLOWED (canonical form):")
for a in sorted(allowed_set):
    print("  ", a)

print("\nDATA VALUES (first 20 distinct):")
print(df_checked[col].drop_duplicates().head(20).to_list())
df_checked.loc[df_checked["P9__invalid"], ["P9", "P9__invalid"]].head(20)

ALLOWED (canonical form):
   [no]
   [unspecified]
   [yes]

DATA VALUES (first 20 distinct):
['[no]', '[unspecified]', '[unspecifiedd]', <NA>]


,P9,P9__invalid
16,[unspecifiedd],True
